#Add Dependencies

In [1]:
import openai
from qdrant_client import QdrantClient

In [2]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [7]:
def retrieve_data(query, collection_name="toys_and_games_small_sample", k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )
    retrieved_context_id = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings=[]

    for result in search_result.points:
        retrieved_context_id.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["all_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_id": retrieved_context_id,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [9]:
retrieved_context=retrieve_data("which item has Bright Red Eyes?")
retrieved_context

{'retrieved_context_id': ['B09X9XW42H',
  'B09NSNDRLL',
  'B0BLHHD8MZ',
  'B0BN7GBJ82',
  'B07KTNZWNK'],
 'retrieved_context': ['Gothic Mothman Plushie Doll with Bright Red Eyes, Medieval Style Attractive Evil Moth Stuffed Plush Doll Toy, Magical Home Decor for Bedroom, for Kids Boyfriend Classmate (A) 🦋 Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. 🦋 This Mothman plushie is looking for a loving and magical home right now. A great gift for your cryptid lover friend or that friend who likes a dark in their home decor. 🦋 It would look great resting on your bed, sitting on your dresser, or on the shelf with your other plushies. 🦋 Material: Plush 🦋 Size: Approx. Height: 8 inches x Width: 9 inches 🦋 Description: Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to joi

### Format retrieved context function

In [36]:
def process_context(retrieved_context):
    formatted_context = []

    for index, (product_id, description, rating, score) in enumerate(
        zip(
            retrieved_context["retrieved_context_id"],
            retrieved_context["retrieved_context"],
            retrieved_context["retrieved_context_ratings"],
            retrieved_context["similarity_scores"],
            strict=True,
        ),
        start=1,
    ):
        formatted_context.append(
            f"Context {index}\n"
            f"Product ID: {product_id}\n"
            f"Rating: {rating}/5\n"
            f"Similarity: {score:.3f}\n"
            f"Description: {description}"
        )

    return "\n\n---\n\n".join(formatted_context)

print(process_context(retrieved_context))

Context 1
Product ID: B09X9XW42H
Rating: 1.3/5
Similarity: 0.424
Description: Gothic Mothman Plushie Doll with Bright Red Eyes, Medieval Style Attractive Evil Moth Stuffed Plush Doll Toy, Magical Home Decor for Bedroom, for Kids Boyfriend Classmate (A) 🦋 Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. 🦋 This Mothman plushie is looking for a loving and magical home right now. A great gift for your cryptid lover friend or that friend who likes a dark in their home decor. 🦋 It would look great resting on your bed, sitting on your dresser, or on the shelf with your other plushies. 🦋 Material: Plush 🦋 Size: Approx. Height: 8 inches x Width: 9 inches 🦋 Description: Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. This Mothman plushie

In [37]:
preprocessed_context=process_context(retrieved_context)

### Create Prompt Template Function

In [39]:
def build_prompt(preprocessed_context, query):
    prompt = f"""
You are a shopping assistant that can answer question about the products in stock.
You will be given a question a nd a list of context.
Instruction:
 - Answer the question based on the context provided onlym
 - Never use word context and refer to it as the available products
 - Do not use markdown formatting.
 
 Context:
{preprocessed_context}
Question:
{query}

 """
    return prompt

In [41]:
prompt = build_prompt(
    preprocessed_context, 
    "Do you have any item with Bright Red Eyes?"
    )
print(prompt)


You are a shopping assistant that can answer question about the products in stock.
You will be given a question a nd a list of context.
Instruction:
 - Answer the question based on the context provided onlym
 - Never use word context and refer to it as the available products
 - Do not use markdown formatting.

 Context:
Context 1
Product ID: B09X9XW42H
Rating: 1.3/5
Similarity: 0.424
Description: Gothic Mothman Plushie Doll with Bright Red Eyes, Medieval Style Attractive Evil Moth Stuffed Plush Doll Toy, Magical Home Decor for Bedroom, for Kids Boyfriend Classmate (A) 🦋 Mothman’s bright red eyes could stare you down in an instant, and his huge dark figure cast fear and doubt in so many residents. He is perfect to join you on your vast adventures. 🦋 This Mothman plushie is looking for a loving and magical home right now. A great gift for your cryptid lover friend or that friend who likes a dark in their home decor. 🦋 It would look great resting on your bed, sitting on your dresser, or 

In [54]:
def generate_asnwer(prompt):
    response = openai.chat.completions.create(
        model = "gpt-6-luna",
        messages=[
            {"role":"system","content":prompt}
        ]
    )

    return response.choices[0].message.content

### Combined RAG pipline

In [55]:
def rag_pipline(query, k=5):
    retrieved_context = retrieve_data(query,collection_name="toys_and_games_small_sample", k=k)
    preprocessed_context=process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_asnwer(prompt)

    return answer

In [57]:
print(rag_pipline("Recommend some toy that can be recharged via USB"))

The available products include two USB-rechargeable options:

• Kidstinct RC Tank (B0BC8TJGQP) — Includes two rechargeable batteries and a USB charger, with up to 60 minutes of playtime. Rated 3.4/5.

• LANSAND RC Excavator (B09XWB3M5X) — Includes two rechargeable batteries and a USB charger, with up to 60 minutes of playtime. Rated 3.8/5.
